<a href="https://colab.research.google.com/github/mauriciodev/progcart/blob/main/aulas/jupyter/duckdb_brasil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install "leafmap[duckdb]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.5/667.5 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 

In [ ]:
url_cidades = "https://metadados.snirh.gov.br/geonetwork/srv/api/records/8db5549f-2b94-40f0-b968-6f940bad68fa/attachments/GEOFT_CIDADE_2016.zip"
url_estados = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2025/Brasil/BR_UF_2025.zip"
db_path = "mydb.duckdb"

In [ ]:
import duckdb
import requests
import zipfile
import io
import geopandas as gpd
import leafmap


# Conexão ao banco de dados em memória

In [ ]:
# conectar duckdb
con = duckdb.connect(db_path) # Objeto que materializa a conexão ao banco de dados.
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")
con

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
!ls -la

total 28
drwxr-xr-x 1 root root  4096 May  3 20:42 .
drwxr-xr-x 1 root root  4096 May  3 20:23 ..
drwxr-xr-x 4 root root  4096 Apr 16 13:28 .config
-rw-r--r-- 1 root root 12288 May  3 20:42 mydb.duckdb
drwxr-xr-x 1 root root  4096 Apr 16 13:28 sample_data


# Cidades


## Criando uma tabela "cidades" temporária (em memória) no DuckDB

In [ ]:
zip_file = leafmap.download_file(url_cidades, unzip=True, overwrite=True)
shp_file = zip_file.replace("zip","shp") # this only works for single layer shapefiles.

Downloading...
From: https://metadados.snirh.gov.br/geonetwork/srv/api/records/8db5549f-2b94-40f0-b968-6f940bad68fa/attachments/GEOFT_CIDADE_2016.zip
To: /content/GEOFT_CIDADE_2016.zip
100%|██████████| 520/520 [00:00<00:00, 1.05MB/s]


BadZipFile: File is not a zip file

In [ ]:
df = con.execute(f"""
CREATE TABLE cidades AS
SELECT *
FROM ST_Read('{shp_file}');
""")

## Lendo a tabela Cidades

In [ ]:
df = con.execute(f""" SELECT * FROM cidades; """).df()
df.head()

In [ ]:
type(df)

# Estados

In [ ]:
zip_file = leafmap.download_file(url_estados, unzip=True, overwrite=True)
shp_file = zip_file.replace("zip","shp") # this only works for single layer shapefiles.

In [ ]:
con.execute(f"""
CREATE TABLE estados AS
SELECT *
FROM ST_Read('{shp_file}');
""")

In [ ]:
df = con.execute(f""" SELECT * FROM estados; """).df()
df.head()

# Mapa

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import leafmap

# cria o mapa
m = leafmap.Map(center=[-14.2350, -51.9253], zoom=4)  # centro aproximado do Brasil

# adiciona os estados (polígonos)
m.add_gdf(
    gdf_estados,
    layer_name="Estados",
    style={
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.2,
    }
)

# adiciona as cidades (pontos)
m.add_gdf(
    gdf_cidades,
    layer_name="Cidades",
    marker_type="circle",   # ou 'marker'
    marker_kwargs={
        "radius": 2,
        "fillOpacity": 0.7,
    },
    cluster=True
)

# controle de camadas
m.add_layer_control()

m

In [ ]:
import leafmap.maplibregl as leafmap
# Initialize the map
m = leafmap.Map()

# Add the DuckDB layer directly
m.add_duckdb_layer(
    database_path=db_path,
    table_name="estados",
    layer_type="fill",
    paint={"fill-color": "#ff0000"},
    opacity=0.5,
    fit_bounds=False,
    src_crs="EPSG:4326",
    quiet=True,
)
m


# Join para contar quantas cidades tem em cada estado

In [ ]:
df = con.execute(f"""
    select *
    from estados
    join cidades
    on CID_SG_UF = SIGLA_UF;
""").df()
df.head()

In [ ]:
df = con.execute(f"""
    select estados.NM_UF, count(*)
    from estados
    join cidades
    on CID_SG_UF = SIGLA_UF
    group by estados.NM_UF;
""").df()
df

In [ ]:
con.execute("Drop table cidades;")

In [ ]:
!ls -la

In [ ]:
%pip install geobr

In [ ]:
from geobr import read_municipality

# Read all municipalities in the country at a given year
mun = read_municipality(code_muni="all", year=2018)

In [ ]:
mun

,code_muni,name_muni,code_state,abbrev_state,geometry
0,1100015.0,Alta Floresta D'oeste,11.0,RO,"MULTIPOLYGON (((-62.23224 -11.90804, -62.2067 ..."
1,1100023.0,Ariquemes,11.0,RO,"MULTIPOLYGON (((-63.57327 -9.78326, -63.57016 ..."
2,1100031.0,Cabixi,11.0,RO,"MULTIPOLYGON (((-60.71834 -13.39058, -60.70904..."
3,1100049.0,Cacoal,11.0,RO,"MULTIPOLYGON (((-61.27873 -11.50596, -61.28097..."
4,1100056.0,Cerejeiras,11.0,RO,"MULTIPOLYGON (((-61.41347 -13.23417, -61.42603..."
...,...,...,...,...,...
5567,5222005.0,Vianópolis,52.0,GO,"POLYGON ((-48.53842 -16.75003, -48.54051 -16.7..."
5568,5222054.0,Vicentinópolis,52.0,GO,"POLYGON ((-50.00189 -17.78179, -50.0142 -17.78..."
5569,5222203.0,Vila Boa,52.0,GO,"POLYGON ((-47.07742 -15.0633, -47.07851 -15.06..."
5570,5222302.0,Vila Propício,52.0,GO,"POLYGON ((-48.91463 -15.20939, -48.91532 -15.1..."
